# CPSC 483 – Programming Assignment 1
## K-Nearest Neighbor Happiness Classifier (From Scratch)

**Dataset:** Fullerton Resident Satisfaction Survey 2015–2020  
**Goal:** Classify residents as Happy (1) or Unhappy (0) using KNN implemented from scratch,  
then compare against Scikit-learn's KNN implementation.

**Features (all rated 1–5):**
1. City Services Availability
2. Housing Cost
3. Quality of Schools
4. Community Trust in Local Police
5. Community Maintenance
6. Availability of Community Room

---

## Imports
The custom KNN, preprocessing, correlation, train-test split, cross-validation, and metrics  
use **only the Python Standard Library** (csv, math, random, collections).  
Scikit-learn is used **only** for the Task 6 comparison. Matplotlib is used for visualisations.

In [ ]:
import csv
import math
import random
from collections import Counter

%matplotlib inline
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

---
## Tasks 1 & 2 – Load Dataset and Move Class Label to Last Column

The original CSV has the class label (`Unhappy/Happy`) as the **first** column.  
The assignment requires it to be the **last** column before any ML is applied.  
The restructured data is also saved to a new CSV file.

In [ ]:
def load_and_restructure(filename):
    """Load CSV, move first column (class label) to last, save restructured file."""
    with open(filename, newline='') as f:
        reader = csv.reader(f)
        rows = list(reader)

    header    = rows[0]
    data_rows = rows[1:]

    # Move first column to last
    new_header = header[1:] + [header[0]]
    new_data   = [row[1:] + [row[0]] for row in data_rows]

    # Save restructured CSV
    out_file = 'HappinessData-1-restructured.csv'
    with open(out_file, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(new_header)
        writer.writerows(new_data)

    print("Original header    :", header)
    print("Restructured header:", new_header)
    print(f"\nRestructured CSV saved → {out_file}")
    return new_header, new_data


FILENAME = 'HappinessData-1.csv'
header, raw_data = load_and_restructure(FILENAME)
print(f"\nDataset: {len(raw_data)} rows × {len(header)} columns")
print("\nFirst 5 rows (raw, restructured):")
for row in raw_data[:5]:
    print(' ', row)

---
## Task 3 – Preprocessing: Handle Missing / NA Values

Some survey questions were not asked in certain years (recorded as `NA`) or received  
no response (recorded as blank). Because features are ordinal (1–5), missing values  
are imputed with the **column mode** (most frequent value), which is appropriate  
for discrete, ordinal data.

In [ ]:
NUM_FEATURES = 6   # columns 0-5 are features; column 6 is the label


def column_mode(data, col_idx):
    """Return the mode of a feature column, ignoring blanks and NA."""
    vals = []
    for row in data:
        v = row[col_idx].strip()
        if v and v.upper() != 'NA':
            vals.append(float(v))
    return Counter(vals).most_common(1)[0][0]


def preprocess(raw_data):
    """Fill missing values with column mode and convert to numeric."""
    modes = [column_mode(raw_data, i) for i in range(NUM_FEATURES)]

    print(f"{'Column':<42s}  {'Mode':>6}  {'Missing':>8}")
    print('-' * 60)
    for i, col_name in enumerate(header[:NUM_FEATURES]):
        n_missing = sum(
            1 for r in raw_data
            if r[i].strip() == '' or r[i].strip().upper() == 'NA'
        )
        print(f"  {col_name:<40s}  {modes[i]:>6}  {n_missing:>8}")

    processed = []
    for row in raw_data:
        new_row = []
        for i in range(NUM_FEATURES):
            v = row[i].strip()
            new_row.append(modes[i] if (v == '' or v.upper() == 'NA') else float(v))
        new_row.append(int(row[NUM_FEATURES].strip()))
        processed.append(new_row)

    total_missing = sum(
        1 for row in raw_data
        for i in range(NUM_FEATURES)
        if row[i].strip() == '' or row[i].strip().upper() == 'NA'
    )
    print(f"\nTotal missing values imputed: {total_missing}")
    return processed


data = preprocess(raw_data)

# Class distribution
class_counts = Counter(row[-1] for row in data)
print(f"\nClass distribution: Unhappy (0)={class_counts[0]}  Happy (1)={class_counts[1]}")

---
## Task 4 – Feature Correlation (Pearson Correlation)

Pearson correlation measures the **linear relationship** between two variables  
and ranges from **−1** (perfect negative) to **+1** (perfect positive).  
We compute:
- Correlation of each feature with the target label
- Feature–feature correlation matrix (with heatmap)

In [ ]:
def pearson(x, y):
    """Pearson correlation coefficient (pure Python)."""
    n = len(x)
    mx, my = sum(x) / n, sum(y) / n
    numerator = sum((x[i] - mx) * (y[i] - my) for i in range(n))
    dx = math.sqrt(sum((v - mx) ** 2 for v in x))
    dy = math.sqrt(sum((v - my) ** 2 for v in y))
    return numerator / (dx * dy) if dx and dy else 0.0


columns    = [[row[i] for row in data] for i in range(NUM_FEATURES + 1)]
target_col = columns[NUM_FEATURES]

# ── Feature vs. target correlation ──
print("Pearson Correlation: each feature vs. target (Happiness)")
print('-' * 56)
feat_target_corr = []
for i in range(NUM_FEATURES):
    r = pearson(columns[i], target_col)
    feat_target_corr.append(r)
    print(f"  {header[i]:<42s}: {r:+.4f}")

# ── Feature–feature correlation matrix (numeric) ──
corr_matrix = [[pearson(columns[i], columns[j])
                for j in range(NUM_FEATURES)]
               for i in range(NUM_FEATURES)]

short = [h[:13] for h in header[:NUM_FEATURES]]
print(f"\n{'':16}", end='')
for s in short:
    print(f"{s:>14}", end='')
print()
for i in range(NUM_FEATURES):
    print(f"{short[i]:16}", end='')
    for j in range(NUM_FEATURES):
        print(f"{corr_matrix[i][j]:>14.3f}", end='')
    print()

In [ ]:
# ── Correlation heatmap ──
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Left: feature vs. target bar chart
ax1 = axes[0]
colors = ['steelblue' if v >= 0 else 'tomato' for v in feat_target_corr]
bars = ax1.barh(range(NUM_FEATURES), feat_target_corr, color=colors, edgecolor='black', linewidth=0.5)
ax1.set_yticks(range(NUM_FEATURES))
ax1.set_yticklabels([h[:22] for h in header[:NUM_FEATURES]], fontsize=9)
ax1.axvline(0, color='black', linewidth=0.8)
ax1.set_xlabel('Pearson r')
ax1.set_title('Feature Correlation with Target (Happiness)', fontsize=11)
for bar, val in zip(bars, feat_target_corr):
    ax1.text(val + (0.01 if val >= 0 else -0.01), bar.get_y() + bar.get_height()/2,
             f'{val:+.3f}', va='center', ha='left' if val >= 0 else 'right', fontsize=8)

# Right: feature-feature heatmap
ax2 = axes[1]
im = ax2.imshow(corr_matrix, cmap='coolwarm', vmin=-1, vmax=1, aspect='auto')
plt.colorbar(im, ax=ax2, fraction=0.046, pad=0.04)
ax2.set_xticks(range(NUM_FEATURES))
ax2.set_yticks(range(NUM_FEATURES))
ax2.set_xticklabels([h[:12] for h in header[:NUM_FEATURES]], rotation=35, ha='right', fontsize=8)
ax2.set_yticklabels([h[:12] for h in header[:NUM_FEATURES]], fontsize=8)
for i in range(NUM_FEATURES):
    for j in range(NUM_FEATURES):
        ax2.text(j, i, f'{corr_matrix[i][j]:.2f}',
                 ha='center', va='center', fontsize=8,
                 color='white' if abs(corr_matrix[i][j]) > 0.5 else 'black')
ax2.set_title('Feature–Feature Correlation Matrix', fontsize=11)

plt.tight_layout()
plt.savefig('correlation_plots.png', dpi=120, bbox_inches='tight')
plt.show()

---
## Task 5 – KNN Algorithm from Scratch

Three distance metrics are implemented and tested:
- **Euclidean** (L2): straight-line distance
- **Manhattan** (L1): sum of absolute differences
- **Minkowski** (p=3): generalisation of the above

Classification rule: among the k nearest neighbours, the **majority class** wins  
(ties broken by `Counter.most_common()` which returns the first encountered).

In [ ]:
# ── Distance metrics ──
def euclidean(a, b):
    return math.sqrt(sum((a[i] - b[i]) ** 2 for i in range(len(a))))

def manhattan(a, b):
    return sum(abs(a[i] - b[i]) for i in range(len(a)))

def minkowski(a, b, p=3):
    return sum(abs(a[i] - b[i]) ** p for i in range(len(a))) ** (1.0 / p)


# ── KNN classifier (from scratch) ──
def knn_predict(train_X, train_y, test_X, k, dist_fn):
    """Predict class labels for all points in test_X."""
    preds = []
    for point in test_X:
        neighbors = sorted(
            ((dist_fn(point, train_X[i]), train_y[i]) for i in range(len(train_X))),
            key=lambda x: x[0]
        )
        votes = [neighbors[j][1] for j in range(k)]
        preds.append(Counter(votes).most_common(1)[0][0])
    return preds


def accuracy(y_true, y_pred):
    return sum(a == b for a, b in zip(y_true, y_pred)) / len(y_true)


print("KNN implementation ready.")
print("Distance metrics: euclidean, manhattan, minkowski(p=3)")

In [ ]:
# ── Train / Test Split – 80 / 20 (from scratch) ──
def split_data(data, test_ratio=0.2, seed=42):
    random.seed(seed)
    d = list(data)
    random.shuffle(d)
    cut = int(len(d) * (1 - test_ratio))
    return d[:cut], d[cut:]


train_data, test_data = split_data(data, test_ratio=0.2, seed=42)
train_X = [r[:-1] for r in train_data]
train_y = [r[-1]  for r in train_data]
test_X  = [r[:-1] for r in test_data]
test_y  = [r[-1]  for r in test_data]

print(f"Train: {len(train_X)} samples   Test: {len(test_X)} samples")
print(f"Train class dist: {Counter(train_y)}")
print(f"Test  class dist: {Counter(test_y)}")

In [ ]:
# ── Compare distance metrics at k=5 ──
print("Custom KNN  k=5 – Distance Metric Comparison")
print(f"{'Metric':<22}  {'Accuracy':>10}  {'Error Rate':>10}")
print('-' * 48)

METRICS = [
    ("Euclidean",      euclidean),
    ("Manhattan",      manhattan),
    ("Minkowski(p=3)", lambda a, b: minkowski(a, b, 3)),
]
metric_results = {}
for name, fn in METRICS:
    preds = knn_predict(train_X, train_y, test_X, 5, fn)
    acc   = accuracy(test_y, preds)
    metric_results[name] = (acc, fn)
    print(f"  {name:<20}  {acc:>10.4f}  {1-acc:>10.4f}")

best_metric_name = max(metric_results, key=lambda k: metric_results[k][0])
best_metric_fn   = metric_results[best_metric_name][1]
print(f"\nBest distance metric: {best_metric_name} "
      f"(Accuracy={metric_results[best_metric_name][0]:.4f})")

---
## Task 6 – Scikit-learn KNN Comparison (k = 5)

We compare our custom KNN against Scikit-learn's `KNeighborsClassifier`  
using the same 80/20 train-test split and k=5.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay

sk5 = KNeighborsClassifier(n_neighbors=5)
sk5.fit(train_X, train_y)
sk5_preds = list(sk5.predict(test_X))
sk5_acc   = accuracy_score(test_y, sk5_preds)

print("── Comparison at k=5 ──")
print(f"{'Model':<30}  {'Accuracy':>10}  {'Error Rate':>10}")
print('-' * 56)
print(f"  {'Custom KNN (Euclidean)':<28}  "
      f"{metric_results['Euclidean'][0]:>10.4f}  "
      f"{1-metric_results['Euclidean'][0]:>10.4f}")
print(f"  {'Custom KNN (Manhattan)':<28}  "
      f"{metric_results['Manhattan'][0]:>10.4f}  "
      f"{1-metric_results['Manhattan'][0]:>10.4f}")
print(f"  {'Scikit-learn KNN':<28}  "
      f"{sk5_acc:>10.4f}  "
      f"{1-sk5_acc:>10.4f}")

---
## Tasks 7 & 8 – Finding Best k: Elbow Plot

We iterate k from 1 to 40, compute the error rate on the test set for both  
the custom KNN (Euclidean) and Scikit-learn KNN, then identify the optimal k  
using the elbow method.

In [ ]:
K_RANGE        = range(1, 41)
custom_errors  = []
sklearn_errors = []

for k in K_RANGE:
    p_custom = knn_predict(train_X, train_y, test_X, k, euclidean)
    custom_errors.append(1 - accuracy(test_y, p_custom))

    sk = KNeighborsClassifier(n_neighbors=k)
    sk.fit(train_X, train_y)
    sklearn_errors.append(1 - accuracy_score(test_y, sk.predict(test_X)))

best_k_custom  = list(K_RANGE)[custom_errors.index(min(custom_errors))]
best_k_sklearn = list(K_RANGE)[sklearn_errors.index(min(sklearn_errors))]

print(f"Best k (Custom KNN)  : {best_k_custom}  (Error={min(custom_errors):.4f})")
print(f"Best k (Scikit-learn): {best_k_sklearn}  (Error={min(sklearn_errors):.4f})")

# Error table for a few key k values
print(f"\n{'k':>4}  {'Custom Error':>14}  {'Sklearn Error':>14}")
print('-' * 38)
for k in [1, 3, 5, 7, 10, best_k_custom, 20, 30, 40]:
    idx = k - 1
    print(f"{k:>4}  {custom_errors[idx]:>14.4f}  {sklearn_errors[idx]:>14.4f}")

In [ ]:
# ── Elbow Plot ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, errors, title in zip(
        axes,
        [custom_errors, sklearn_errors],
        ['Custom KNN (Euclidean)', 'Scikit-learn KNN']):
    ax.plot(list(K_RANGE), errors, 'b--o',
            markerfacecolor='red', markersize=6, linewidth=1.2)
    best_k = list(K_RANGE)[errors.index(min(errors))]
    ax.axvline(best_k, color='green', linestyle=':', linewidth=1.5,
               label=f'Best k={best_k}')
    ax.set_title(f'Error Rate vs. K Value\n({title})', fontsize=12)
    ax.set_xlabel('k value')
    ax.set_ylabel('Error Rate')
    ax.set_xticks(range(0, 41, 5))
    ax.grid(True, alpha=0.3)
    ax.legend()

plt.tight_layout()
plt.savefig('elbow_plot.png', dpi=120, bbox_inches='tight')
plt.show()

---
## Task 9 – N-Fold Cross-Validation (from scratch)

Cross-validation provides a more robust performance estimate by rotating  
through different train/test splits. We implement **5-fold CV** from scratch  
and test it for several k values.

In [ ]:
def cross_validate(data, n_folds, k, dist_fn, seed=42):
    """N-fold cross-validation. Returns (mean_acc, std, fold_accs)."""
    random.seed(seed)
    d = list(data)
    random.shuffle(d)

    fold_sz = len(d) // n_folds
    folds   = [d[i * fold_sz:(i + 1) * fold_sz] for i in range(n_folds)]
    # distribute any remainder rows across folds
    for idx, row in enumerate(d[n_folds * fold_sz:]):
        folds[idx % n_folds].append(row)

    fold_accs = []
    for i in range(n_folds):
        test_fold  = folds[i]
        train_fold = [r for j, fold in enumerate(folds) if j != i for r in fold]
        tX = [r[:-1] for r in train_fold];  ty = [r[-1] for r in train_fold]
        eX = [r[:-1] for r in test_fold];   ey = [r[-1] for r in test_fold]
        preds = knn_predict(tX, ty, eX, k, dist_fn)
        fold_accs.append(accuracy(ey, preds))

    mean = sum(fold_accs) / len(fold_accs)
    std  = math.sqrt(sum((a - mean) ** 2 for a in fold_accs) / len(fold_accs))
    return mean, std, fold_accs


# ── 5-Fold CV results ──
print("5-Fold Cross-Validation  –  Custom KNN, Euclidean Distance")
print(f"{'k':>4}  {'Mean Acc':>10}  {'Std Dev':>8}  Fold Accuracies")
print('-' * 72)

cv_means, cv_ks = [], []
for k in sorted({1, 3, 5, 7, best_k_custom, 10, 15}):
    mean, std, fold_accs = cross_validate(data, 5, k, euclidean)
    cv_means.append(mean)
    cv_ks.append(k)
    folds_str = '  '.join(f'{a:.3f}' for a in fold_accs)
    print(f"{k:>4}  {mean:>10.4f}  {std:>8.4f}  [{folds_str}]")

best_cv_k = cv_ks[cv_means.index(max(cv_means))]
print(f"\nBest k by cross-validation: {best_cv_k} (Mean Acc={max(cv_means):.4f})")

In [ ]:
# ── CV accuracy plot across k values ──
cv_all_means = []
for k in K_RANGE:
    mean, _, _ = cross_validate(data, 5, k, euclidean)
    cv_all_means.append(mean)

best_cv_k_all = list(K_RANGE)[cv_all_means.index(max(cv_all_means))]

plt.figure(figsize=(10, 4))
plt.plot(list(K_RANGE), cv_all_means, 'g--s',
         markerfacecolor='orange', markersize=5, linewidth=1.2,
         label='5-Fold CV Mean Accuracy')
plt.axvline(best_cv_k_all, color='red', linestyle=':', linewidth=1.5,
            label=f'Best k={best_cv_k_all} (Acc={max(cv_all_means):.3f})')
plt.title('5-Fold Cross-Validation Accuracy vs. K (Custom KNN, Euclidean)', fontsize=12)
plt.xlabel('k value')
plt.ylabel('Mean Accuracy')
plt.xticks(range(0, 41, 5))
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig('cv_accuracy_plot.png', dpi=120, bbox_inches='tight')
plt.show()

print(f"Best k by 5-fold CV (k=1..40): {best_cv_k_all}  "
      f"(Mean Acc={max(cv_all_means):.4f})")

---
## Task 10 – Classification Metrics & Confusion Matrix

All metrics are computed **from scratch** without using sklearn metrics functions.  
Metrics reported:
- **Accuracy** = (TP + TN) / Total
- **Precision** = TP / (TP + FP)
- **Recall (Sensitivity)** = TP / (TP + FN)
- **F1 Score** = 2 × Precision × Recall / (Precision + Recall)
- **Specificity** = TN / (TN + FP)

In [ ]:
def binary_metrics(y_true, y_pred, label=''):
    """Compute and print full set of classification metrics from scratch."""
    tp = sum(1 for t, p in zip(y_true, y_pred) if t == 1 and p == 1)
    fp = sum(1 for t, p in zip(y_true, y_pred) if t == 0 and p == 1)
    fn = sum(1 for t, p in zip(y_true, y_pred) if t == 1 and p == 0)
    tn = sum(1 for t, p in zip(y_true, y_pred) if t == 0 and p == 0)

    total = tp + fp + fn + tn
    acc   = (tp + tn) / total       if total        else 0.0
    prec  = tp / (tp + fp)          if (tp + fp)    else 0.0
    rec   = tp / (tp + fn)          if (tp + fn)    else 0.0
    f1    = 2*prec*rec/(prec+rec)   if (prec + rec) else 0.0
    spec  = tn / (tn + fp)          if (tn + fp)    else 0.0

    bar = '=' * 56
    print(f"\n{bar}")
    print(f"  {label}")
    print(bar)
    print(f"  Confusion Matrix (rows=Actual, cols=Predicted):")
    print(f"  {'':<24}  {'Pred 0':>8}  {'Pred 1':>8}")
    print(f"  {'Actual 0 (Unhappy)':<24}  {tn:>8}  {fp:>8}")
    print(f"  {'Actual 1 (Happy)':<24}  {fn:>8}  {tp:>8}")
    print(f"\n  Accuracy    : {acc:.4f}")
    print(f"  Precision   : {prec:.4f}  (predicted Happy → truly Happy)")
    print(f"  Recall      : {rec:.4f}  (truly Happy → correctly caught)")
    print(f"  F1 Score    : {f1:.4f}")
    print(f"  Specificity : {spec:.4f}  (truly Unhappy → correctly caught)")
    return dict(tp=tp, fp=fp, fn=fn, tn=tn,
                accuracy=acc, precision=prec, recall=rec, f1=f1, specificity=spec)


# ── Evaluate: Custom KNN – best k (Euclidean) ──
best_preds_custom = knn_predict(train_X, train_y, test_X, best_k_custom, euclidean)
m1 = binary_metrics(test_y, best_preds_custom,
                    f'Custom KNN  |  k={best_k_custom}  |  Euclidean  |  Test Set')

# ── Evaluate: Custom KNN – k=5 Manhattan ──
preds_manhattan = knn_predict(train_X, train_y, test_X, 5, manhattan)
m2 = binary_metrics(test_y, preds_manhattan,
                    'Custom KNN  |  k=5  |  Manhattan  |  Test Set')

# ── Evaluate: Scikit-learn – best k ──
sk_best = KNeighborsClassifier(n_neighbors=best_k_sklearn)
sk_best.fit(train_X, train_y)
sk_best_preds = list(sk_best.predict(test_X))
m3 = binary_metrics(test_y, sk_best_preds,
                    f'Scikit-learn KNN  |  k={best_k_sklearn}  |  Test Set')

In [ ]:
# ── Scikit-learn full classification report (k=5, for reference) ──
print("Scikit-learn Classification Report  (k=5)")
print(classification_report(test_y, sk5_preds,
                             target_names=['Unhappy (0)', 'Happy (1)']))

In [ ]:
# ── Confusion matrix visualisations ──
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

configs = [
    (test_y, best_preds_custom, f'Custom KNN\n(k={best_k_custom}, Euclidean)'),
    (test_y, preds_manhattan,   'Custom KNN\n(k=5, Manhattan)'),
    (test_y, sk_best_preds,     f'Scikit-learn KNN\n(k={best_k_sklearn})'),
]
for ax, (yt, yp, title) in zip(axes, configs):
    ConfusionMatrixDisplay.from_predictions(
        yt, yp,
        display_labels=['Unhappy', 'Happy'],
        cmap='Blues', ax=ax
    )
    ax.set_title(title, fontsize=11)

plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ── Summary comparison table ──
print("="*70)
print("  FINAL SUMMARY – Model Comparison")
print("="*70)
print(f"{'Model':<38}  {'Acc':>6}  {'Prec':>6}  {'Rec':>6}  {'F1':>6}")
print('-'*70)
rows_summary = [
    (f'Custom KNN k={best_k_custom} Euclidean (test)', m1),
    ('Custom KNN k=5 Manhattan (test)',                 m2),
    (f'Scikit-learn k={best_k_sklearn} (test)',         m3),
]
for label, m in rows_summary:
    print(f"  {label:<36}  {m['accuracy']:>6.3f}  {m['precision']:>6.3f}  "
          f"{m['recall']:>6.3f}  {m['f1']:>6.3f}")

# Cross-validation row
cv_mean, cv_std, _ = cross_validate(data, 5, best_cv_k_all, euclidean)
print(f"\n  Custom KNN k={best_cv_k_all} Euclidean (5-fold CV mean): "
      f"Accuracy = {cv_mean:.4f} ± {cv_std:.4f}")
print("="*70)